<a href="https://colab.research.google.com/github/MatheusRegoDev/AMnS-projects-2025.2-BTI-UFRN/blob/main/Treinamento_de_Classificador_de_Texto_com_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial - Treinamento de Classificador de Texto com Embeddings

# 1. Introdução a utilização de embeddings

In [ ]:
import tensorflow as tf
import tensorflow_text
import tensorflow_hub as hub
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd


embed_model = hub.load("https://www.kaggle.com/models/google/universal-sentence-encoder/tensorFlow2/multilingual/2")

def embed_texts(texts):
    """Converte uma lista de textos em embeddings (vetores numéricos)."""
    return embed_model(texts).numpy()

def compute_similarity(text1, text2):
    """Calcula a similaridade cosseno entre dois textos."""
    embeddings = embed_texts([text1, text2])
    return cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]

np.float32(0.7166079)

In [ ]:
# ==========================================
# EXEMPLO 1: Avaliação de Qualidade de Mensagens em Chat
# ==========================================

# Keywords definidas pelo professor sobre o tema
keywords_professor = "enzimas digestão sistema digestivo proteases lipases amilases boca estômago intestino"

# Mensagens de alunos (algumas boas, outras ruins)
mensagens_alunos = [
    "As enzimas quebram as moléculas de comida em partes menores",  # BOA
    "O sistema digestivo usa enzimas para digerir proteínas e gorduras",  # BOA
    "Enzimas como amilase e protease atuam no estômago e intestino",  # BOA
    "Eu não entendi nada dessa aula",  # RUIM
    "Que legal, vou estudar em casa"  # RUIM (genérica, sem conteúdo)
]

# Calcula embeddings
print("Calculando embeddings das mensagens...")
embeddings_mensagens = embed_texts(mensagens_alunos)
embeddings_keywords = embed_texts([keywords_professor])

# Calcula similaridade de cada mensagem com os keywords
print(f"\nKeywords do professor:\n'{keywords_professor}'\n")
print("-" * 70)
print(f"{'Mensagem do Aluno':<50} | Similaridade")
print("-" * 70)

similaridades = []
for i, msg in enumerate(mensagens_alunos):
    sim = cosine_similarity([embeddings_mensagens[i]], embeddings_keywords[0].reshape(1, -1))[0][0]
    similaridades.append(sim)
    # Classifica como boa ou ruim baseado na similaridade
    print(f"{msg:<50} | {sim:.3f}")

Calculando embeddings das mensagens...

Keywords do professor:
'enzimas digestão sistema digestivo proteases lipases amilases boca estômago intestino'

----------------------------------------------------------------------
Mensagem do Aluno                                  | Similaridade
----------------------------------------------------------------------
As enzimas quebram as moléculas de comida em partes menores | 0.484
O sistema digestivo usa enzimas para digerir proteínas e gorduras | 0.692
Enzimas como amilase e protease atuam no estômago e intestino | 0.618
Eu não entendi nada dessa aula                     | 0.029
Que legal, vou estudar em casa                     | -0.026


In [ ]:

# ==========================================
# EXEMPLO 2: Multilíngue - Mesma Mensagem em Diferentes Idiomas
# ==========================================

# Pega a primeira mensagem "boa" e traduz para outros idiomas
mensagem_original = "As enzimas quebram as moléculas de comida em partes menores"

# Traduções (você pode usar Google Translate ou fazer manualmente)
mensagem_en = "Enzymes break down food molecules into smaller parts"
mensagem_es = "Las enzimas descomponen las moléculas de alimentos en partes más pequeñas"
mensagem_zh = "酶将食物分子分解成更小的部分"  # Chinês simplificado

mensagens_multilingues = {
    "Português (Original)": mensagem_original,
    "English": mensagem_en,
    "Español": mensagem_es,
    "中文 (Chinês)": mensagem_zh
}

print(f"Mensagem Original (PT-BR):\n'{mensagem_original}'\n")
print("-" * 70)
print("Calculando embeddings em diferentes idiomas...")

# Calcula embeddings para todas as versões
embeddings_multi = embed_texts(list(mensagens_multilingues.values()))

# Cria matriz de similaridade
matriz_similaridade = cosine_similarity(embeddings_multi)

# Exibe a matriz de forma legível
idiomas = list(mensagens_multilingues.keys())
df_similaridade = pd.DataFrame(
    matriz_similaridade,
    index=idiomas,
    columns=idiomas
)

print(df_similaridade.round(3))

# Mostra a similaridade de cada tradução com o original
print("-" * 70)
print("Similaridade de cada tradução com o original (PT-BR):")
print("-" * 70)
for i, idioma in enumerate(idiomas[1:], 1):
    sim = matriz_similaridade[0][i]
    print(f"{idioma:<20} → Similaridade: {sim:.3f}")

print("💡 Interpretação: Valores próximos de 1.0 indicam que o significado é")
print("   preservado mesmo em idiomas diferentes!")

Mensagem Original (PT-BR):
'As enzimas quebram as moléculas de comida em partes menores'

----------------------------------------------------------------------
Calculando embeddings em diferentes idiomas...
                      Português (Original)  English  Español  中文 (Chinês)
Português (Original)                 1.000    0.892    0.919        0.805
English                              0.892    1.000    0.898        0.867
Español                              0.919    0.898    1.000        0.831
中文 (Chinês)                          0.805    0.867    0.831        1.000
----------------------------------------------------------------------
Similaridade de cada tradução com o original (PT-BR):
----------------------------------------------------------------------
English              → Similaridade: 0.892
Español              → Similaridade: 0.919
中文 (Chinês)          → Similaridade: 0.805
💡 Interpretação: Valores próximos de 1.0 indicam que o significado é
   preservado mesmo em idiom

# 2. Treinando um Classificador de Texto com Embeddings Pré-treinados

### Código-fonte

In [ ]:
import tensorflow as tf
import tensorflow_text
import tensorflow_hub as hub
from tensorflow.keras import regularizers
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import yaml
import wandb
from wandb.integration.keras import WandbMetricsLogger
from sklearn.model_selection import train_test_split
import pandas as pd

def build_encoder_layer(hub_url="https://www.kaggle.com/models/google/universal-sentence-encoder/tensorFlow2/multilingual/2"):
    """Camada Keras para o Universal Sentence Encoder."""
    hub_module = hub.load(hub_url)
    class HubLayer(tf.keras.layers.Layer):
        def __init__(self, **kwargs):
            super().__init__(**kwargs)
            self.hub_module = hub_module
            self.hub_module.trainable = False
        def call(self, inputs):
            return self.hub_module(inputs)
    return HubLayer()

class SimpleIntentClassifier:
    """
    Classificador de Intenções
    """
    def __init__(self, config=None):
        """
        Inicializa o classificador com hiperparâmetros configuráveis.
        Se `config` for None, usa valores padrão. Isso facilita a integração com W&B Sweeps.
        """
        # Valores padrão para hiperparâmetros
        default_config = {
            "hidden_units": 32,
            "dropout_rate": 0.1,
            "l1_reg": 0.01,
            "l2_reg": 0.01,
            "learning_rate": 0.005,
            "epochs": 50,
            "batch_size": 16,
            "patience": 10
        }

        # Atualiza os padrões com o que foi passado (útil para Sweeps)
        self.config = default_config
        if config:
            self.config.update(config)

        self.model = None
        self.encoder = None
        self.classes = None

    def prepare_data(self, texts, labels):
        """Prepara dados (One-Hot Encoding para labels e Tensores para texto)."""
        self.classes = np.unique(labels)
        self.encoder = OneHotEncoder(categories=[self.classes], sparse_output=False)
        y_encoded = self.encoder.fit_transform(np.array(labels).reshape(-1, 1))
        X_tensor = tf.convert_to_tensor(texts, dtype=tf.string)
        return X_tensor, y_encoded

    def build_model(self, num_classes):
        """Constrói a arquitetura usando os hiperparâmetros da configuração."""
        text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name="entrada_texto")
        encoded_text = build_encoder_layer()(text_input)

        # Usa os parâmetros da config (hidden_units, l1_reg, l2_reg)
        hidden = tf.keras.layers.Dense(
            units=self.config["hidden_units"],
            activation='relu',
            kernel_regularizer=regularizers.l1_l2(l1=self.config["l1_reg"], l2=self.config["l2_reg"]),
            name='camada_oculta'
        )(encoded_text)

        hidden = tf.keras.layers.BatchNormalization()(hidden)
        # Usa o parâmetro de dropout
        hidden = tf.keras.layers.Dropout(self.config["dropout_rate"])(hidden)

        output = tf.keras.layers.Dense(
            units=num_classes,
            activation='softmax',
            name='saida_probabilidades'
        )(hidden)

        model = tf.keras.Model(inputs=text_input, outputs=output)

        # Usa o learning_rate da config
        model.compile(
            loss='categorical_crossentropy',
            optimizer=tf.keras.optimizers.Adam(learning_rate=self.config["learning_rate"]),
            metrics=['accuracy']
        )
        return model

    def train(self, texts, labels, wandb_project=None, test_size=None):
        """
        Treina o modelo. inicializa e loga tudo no W&B.

        Args:
            texts: Lista de textos
            labels: Lista de rótulos
            wandb_project: Nome do projeto W&B
            test_size: Proporção para test set (ex: 0.2 para 80/20 split).
                      Se None, usa validation_split durante o fit.
        """
        X, y = self.prepare_data(texts, labels)
        self.model = self.build_model(num_classes=len(self.classes))

        # Se test_size for fornecido, faz train/test split
        if test_size is not None:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=test_size, random_state=42, stratify=np.argmax(y, axis=1)
            )
        else:
            X_train, y_train = X, y
            X_test, y_test = None, None

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=self.config["patience"],
                restore_best_weights=True
            )
        ]

        # Integração com W&B
        if wandb_project:
            wandb.init(project=wandb_project, config=self.config)
            self.config = wandb.config
            callbacks.append(WandbMetricsLogger())

        # Se há test set, usa como validation; senão usa validation_split
        if X_test is not None:
            validation_data = (X_test, y_test)
            validation_split = None
        else:
            validation_data = None
            validation_split = 0.2

        history = self.model.fit(
            X_train, y_train,
            epochs=self.config["epochs"],
            batch_size=self.config["batch_size"],
            validation_data=validation_data,
            validation_split=validation_split,
            callbacks=callbacks,
            verbose=1
        )

        if wandb_project and wandb.run is not None:
            wandb.finish()

        return history

    def predict(self, text):
        """Faz a previsão para um novo texto."""
        if self.model is None:
            raise ValueError("Treine o modelo antes de fazer previsões.")
        text_tensor = tf.convert_to_tensor([text], dtype=tf.string)
        probs = self.model.predict(text_tensor, verbose=0)[0]
        best_idx = np.argmax(probs)
        best_class = self.classes[best_idx]
        confidence = probs[best_idx]

        # Retorna também todas as probabilidades
        all_probs = {self.classes[i]: float(probs[i]) for i in range(len(self.classes))}

        return best_class, confidence, all_probs

    def evaluate(self, texts, labels):
        """
        Avalia o modelo em um conjunto de dados (ex: test set).

        Args:
            texts: Lista de textos
            labels: Lista de rótulos

        Returns:
            dict: Dicionário com loss e accuracy
        """
        if self.model is None:
            raise ValueError("Treine o modelo antes de avaliar.")

        X, y = self.prepare_data(texts, labels)
        results = self.model.evaluate(X, y, verbose=0)

        return {
            'loss': results[0],
            'accuracy': results[1]
        }

### Exemplo 1 - Testes básicos

In [ ]:
# Exemplo 1

textos = ["olá", "bom dia", "cancelar conta", "fechar", "qual o saldo", "ver saldo"]
labels = ["saudacao", "saudacao", "cancelamento", "cancelamento", "saldo", "saldo"]

clf = SimpleIntentClassifier()

clf.train(textos, labels)

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - accuracy: 0.5000 - loss: 10.4006 - val_accuracy: 0.0000e+00 - val_loss: 9.9831
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - accuracy: 1.0000 - loss: 9.0487 - val_accuracy: 0.0000e+00 - val_loss: 9.5694
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - accuracy: 1.0000 - loss: 8.5956 - val_accuracy: 0.0000e+00 - val_loss: 9.1406
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 1.0000 - loss: 8.1288 - val_accuracy: 0.0000e+00 - val_loss: 8.7110
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 1.0000 - loss: 7.5873 - val_accuracy: 0.0000e+00 - val_loss: 8.2776
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - accuracy: 1.0000 - loss: 7.1041 - val_accuracy: 0.0000e+00 - val_loss: 7.8466
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 1.0000 - loss: 6.6493 - val_accuracy: 0.0000e+00 - val_loss: 7.4305
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 1.0000 - loss: 6.2452 - val_

In [ ]:
clf.predict("oi")

(np.str_('saudacao'),
 np.float32(0.526313),
 {np.str_('cancelamento'): 0.35353320837020874,
  np.str_('saldo'): 0.12015382945537567,
  np.str_('saudacao'): 0.5263130068778992})

### Exemplo 2 - Classificador de confusão

Carregue os dados fornecidos em formato `.yml` fornecidos na tarefa.

In [ ]:
def load_intents_from_yaml(yaml_path):
    """
    Carrega dados de um arquivo YAML e os transforma no formato esperado pelo SimpleIntentClassifier.

    Formato esperado do YAML:
    - intent: nome_da_intencao
      examples:
        - exemplo 1
        - exemplo 2
    - intent: outra_intencao
      examples:
        - exemplo 3
        - exemplo 4

    Args:
        yaml_path: Caminho para o arquivo YAML

    Returns:
        tuple: (textos, labels) onde:
            - textos: Lista de strings (exemplos)
            - labels: Lista de strings (intenções correspondentes)

    Exemplo:
        textos, labels = load_intents_from_yaml('confusion_examples.yml')
    """
    with open(yaml_path, 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)

    textos = []
    labels = []

    # Itera sobre cada intenção no arquivo
    for intent_block in data:
        intent_name = intent_block['intent']
        examples = intent_block['examples']

        # Adiciona cada exemplo com seu rótulo correspondente
        for example in examples:
            textos.append(example)
            labels.append(intent_name)

    return textos, labels

In [ ]:
# Passo 1: Carregar os dados do arquivo YAML
print("Carregando dados de confusion_examples.yml...")
textos, labels = load_intents_from_yaml('confusion_examples.yml')
textos = [str(t) for t in textos]

print(f"✓ Carregados {len(textos)} exemplos")
print(f"✓ Intenções encontradas: {np.unique(labels)}")

Carregando dados de confusion_examples.yml...
✓ Carregados 193 exemplos
✓ Intenções encontradas: ['certainty' 'confusion' 'neutral_statement']


In [ ]:
# Passo 2: Fazer train/test split (80/20)
print("Fazendo train/test split (80/20)...")
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    textos, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f"✓ Train set: {len(X_train)} exemplos")
print(f"✓ Test set: {len(X_test)} exemplos")

Fazendo train/test split (80/20)...
✓ Train set: 154 exemplos
✓ Test set: 39 exemplos


In [ ]:
# Passo 3: Criar o classificador com configuração customizada
config = {
    "epochs": 100,
    "hidden_units": 32,
    "learning_rate": 0.001,
    "patience": 15
}

clf = SimpleIntentClassifier(config=config)

# Passo 4: Treinar o modelo com o train set
print("Iniciando treinamento...")
history = clf.train(X_train, y_train)

Iniciando treinamento...
Epoch 1/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 11s 273ms/step - accuracy: 0.4715 - loss: 9.9618 - val_accuracy: 0.4839 - val_loss: 9.5885
Epoch 2/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.7480 - loss: 8.8777 - val_accuracy: 0.6129 - val_loss: 8.8672
Epoch 3/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.8455 - loss: 8.0316 - val_accuracy: 0.6774 - val_loss: 8.1381
Epoch 4/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.9350 - loss: 7.2266 - val_accuracy: 0.6774 - val_loss: 7.4155
Epoch 5/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.9268 - loss: 6.4937 - val_accuracy: 0.6452 - val_loss: 6.7184
Epoch 6/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.9431 - loss: 5.7832 - val_accuracy: 0.6452 - val_loss: 6.0571
Epoch 7/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.9593 - loss: 5.1225 - val_accuracy: 0.5806 - val_loss: 5.4335
Epoch 8/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9675 - loss: 4.5162 - val_a

In [ ]:
clf.predict("estou confuso")

(np.str_('neutral_statement'),
 np.float32(0.4073664),
 {np.str_('certainty'): 0.20351365208625793,
  np.str_('confusion'): 0.38911986351013184,
  np.str_('neutral_statement'): 0.40736639499664307})

In [ ]:
# Passo 5: Avaliar no test set
print("\n" + "="*70)
print("AVALIAÇÃO NO TEST SET")
print("="*70)
test_metrics = clf.evaluate(X_test, y_test)
print(f"Loss: {test_metrics['loss']:.4f}")
print(f"Accuracy: {test_metrics['accuracy']:.2%}")


AVALIAÇÃO NO TEST SET
Loss: 0.8700
Accuracy: 76.92%


In [ ]:
clf.predict("tenho certeza que estou confuso")

(np.str_('confusion'),
 np.float32(0.88515466),
 {np.str_('certainty'): 0.08746929466724396,
  np.str_('confusion'): 0.885154664516449,
  np.str_('neutral_statement'): 0.02737610973417759})

In [ ]:
# Passo 6: Fazer previsões em exemplos individuais
print("="*70)
print("TESTANDO O MODELO COM EXEMPLOS")
print("="*70)

test_phrases = [
    "espera, oque?",
    "tou perdido",
    "agora eu sei exatamente o que fazer",
    "isso faz todo sentido",
    "tudo ok, vamos seguir"
]

for phrase in test_phrases:
    intencao, confianca, probs = clf.predict(phrase)
    print(f"\nFrase: '{phrase}'")
    print(f"Intenção: {intencao} (Confiança: {confianca:.1%})")
    print(f"Probabilidades: {probs}")

TESTANDO O MODELO COM EXEMPLOS

Frase: 'espera, oque?'
Intenção: confusion (Confiança: 95.5%)
Probabilidades: {np.str_('certainty'): 0.006109094712883234, np.str_('confusion'): 0.9553179740905762, np.str_('neutral_statement'): 0.03857286274433136}

Frase: 'tou perdido'
Intenção: confusion (Confiança: 95.3%)
Probabilidades: {np.str_('certainty'): 0.010639901272952557, np.str_('confusion'): 0.9529781341552734, np.str_('neutral_statement'): 0.03638198599219322}

Frase: 'agora eu sei exatamente o que fazer'
Intenção: confusion (Confiança: 92.4%)
Probabilidades: {np.str_('certainty'): 0.04291795566678047, np.str_('confusion'): 0.9237146973609924, np.str_('neutral_statement'): 0.03336739167571068}

Frase: 'isso faz todo sentido'
Intenção: certainty (Confiança: 83.1%)
Probabilidades: {np.str_('certainty'): 0.8306258320808411, np.str_('confusion'): 0.09502170234918594, np.str_('neutral_statement'): 0.07435250282287598}

Frase: 'tudo ok, vamos seguir'
Intenção: neutral_statement (Confiança: 52.

### Exemplo 3 - Intents da Clair